# Wine Type Prediction Using a Neural Network
Beginner-friendly walkthrough: prepare red/white labels, explore the data, train a neural network, evaluate it, and try a new sample.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_DIR = Path('data') if Path('data/redwinequality.csv').exists() else Path('../data')
if not (DATA_DIR / 'redwinequality.csv').exists():
    try:
        from google.colab import files
        print('Upload redwinequality.csv and whitewinequality.csv')
        files.upload()
        DATA_DIR = Path('.')
    except ImportError:
        raise FileNotFoundError('Put both CSV files in data/ or upload them in Colab.')

In [ ]:
red = pd.read_csv(DATA_DIR / 'redwinequality.csv', sep=';')
white = pd.read_csv(DATA_DIR / 'whitewinequality.csv', sep=';')
red['type'] = 1  # red wine
white['type'] = 0  # white wine
wines = pd.concat([red, white], ignore_index=True).dropna()
print(f'Red samples: {len(red):,}; white samples: {len(white):,}; total: {len(wines):,}')
wines.head()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(wines.loc[wines.type == 1, 'alcohol'], bins=20, alpha=.6, label='Red wine')
plt.hist(wines.loc[wines.type == 0, 'alcohol'], bins=20, alpha=.6, label='White wine')
plt.xlabel('Alcohol (% by volume)'); plt.ylabel('Number of samples')
plt.title('Alcohol distribution by wine type'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
FEATURES = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
X, y = wines[FEATURES], wines['type']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=42, stratify=y)
model = Pipeline([('scaler', StandardScaler()), ('neural_network', MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42, early_stopping=True))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, pred):.2%}')
print(classification_report(y_test, pred, target_names=['White wine', 'Red wine'], zero_division=0))
print('Confusion matrix [white, red]:'); print(confusion_matrix(y_test, pred))

In [ ]:
preview = X_test.head(12).copy()
preview['actual_type'] = y_test.head(12).map({0: 'White wine', 1: 'Red wine'})
preview['predicted_type'] = pd.Series(model.predict(X_test.head(12)), index=X_test.head(12).index).map({0: 'White wine', 1: 'Red wine'})
preview['red_wine_probability'] = model.predict_proba(X_test.head(12))[:, 1]
preview['correct'] = preview.actual_type == preview.predicted_type
preview

## Try a new sample
The measurements below are illustrative; replace them with the measurements of a sample you want to classify.

In [ ]:
new_wine = pd.DataFrame([{'fixed acidity':7.0, 'volatile acidity':0.27, 'citric acid':0.36, 'residual sugar':2.0, 'chlorides':0.045, 'free sulfur dioxide':30.0, 'total sulfur dioxide':120.0, 'density':0.995, 'pH':3.2, 'sulphates':0.50, 'alcohol':10.0, 'quality':6}], columns=FEATURES)
label = int(model.predict(new_wine)[0]); probability = float(model.predict_proba(new_wine)[0, 1])
print('Predicted type:', 'Red wine' if label == 1 else 'White wine')
print(f'Estimated red-wine probability: {probability:.1%}')

## Notes
- This predicts wine type, not wine quality.
- Accuracy is measured on a held-out test set and is not a guarantee for every future sample.
- `quality` is included as an input feature; remove it from `FEATURES` to compare performance without it.